# Experiment 024: doubled-budget supervised controls at 7B (Colab)

The final arm of the selected 7B package. Two supervised variants train on the
same 245 MMLU prompts for 16 passes, which is 245 optimiser steps at effective
batch 16:

- `024_p16`: MC-SFT from the raw 7B base, seeds 0/1/2;
- `024_wsft_p16`: MC-SFT from each seed's matching 013 GSM8K-SFT base, seeds 0/1/2.

Questions were fixed before the 022 results: does doubling the supervised
budget change the 014 endpoint, and where does starting-point-matched MC-SFT
finish beside smooth-1000 GRPO? This is a relative supervised budget control,
not a compute-matched or update-matched comparison with GRPO. Like earlier
exposure curves, p16 has its own linear decay over 245 steps, so this is the
training budget as practised rather than a data-exposure-only comparison. The
probe is checked again to separate answer expression from knowledge change.

Execution is staged for a technical check only. Seed 0 runs first, then the seed
list is widened to all three. Scientific results from seed 0 do not decide
whether seeds 1 and 2 run. Any failure stops the sweep without changing batch,
accumulation, checkpointing or the disabled fallback ladder.

The completed 014, 015 and 022 artifacts are references only and are never
modified. Expected cost is about 4 hours and 25 compute units.


## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` before starting the GPU work.


In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 79.83   # fill from the resources panel
unit_rate = 6.77   # fill from the resources panel
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '024_colab_b7_mc_sft_p16',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260801_164657 | cu_before = 79.83
Mounted at /content/drive
closed previous session: 20260801_123814
saved open session: 20260801_164657
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required


In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b supervised runs'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-929869db-cd09-29cc-ebb5-70199ea7cb3b)
total vram: 79.3 GiB


## 2. Clone or update the repo


In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 2915, done.
remote: Counting objects: 100% (399/399), done.
remote: Compressing objects: 100% (252/252), done.
remote: Total 2915 (delta 246), reused 294 (delta 147), pack-reused 2516 (from 1)
Receiving objects: 100% (2915/2915), 83.15 MiB | 23.86 MiB/s, done.
Resolving deltas: 100% (1865/1865), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies


In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# colab includes a torchao version that is incompatible with this peft version
# no quantisation is used in these runs
!pip uninstall -q -y torchao
print('dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 160.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.9 MB/s eta 0:00:00
dependencies installed


## 4. W&B key

Grant this notebook access to `WANDB_API_KEY` in Colab secrets.
The prompt is used only when the W&B secret is unavailable. This key is only used for experiment tracking.


In [5]:
import getpass, os
wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None
if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')
os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check Drive space

The warm arms reuse each 013 run's existing merged base, so this notebook does
not create three more full 7B model copies.


In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 40, 'need at least 40GB free on drive for the six supervised arms'


open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260801_164657.json
drive free: 179 GB


## 6. Build the data splits


In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 19.6MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 3.05MB/s,  227kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 3.05MB/s,  228kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 693kB/s, 41.5kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 693kB/s, 41.5kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 377084.69 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 273374.86 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 87.8MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 133MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Select the staged arms

Keep seed 0 for the first pass. After its technical checks complete, I should use the
three-seed line and run the notebook again. Completed work skips automatically.


In [8]:
# SEEDS_TO_RUN = (0,)
SEEDS_TO_RUN = (0, 1, 2)
VARIANTS = ('base', 'warm')

def config_path(variant, seed):
    prefix = '024_p16' if variant == 'base' else '024_wsft_p16'
    return f'configs/{prefix}_colab_b7_seed{seed}.yaml'

def run_tag(variant, seed):
    prefix = '024_p16' if variant == 'base' else '024_wsft_p16'
    return f'{prefix}_COLAB_FULL_B7_sft_only_seed{seed}'

JOBS = [(variant, seed, config_path(variant, seed), run_tag(variant, seed))
        for seed in SEEDS_TO_RUN for variant in VARIANTS]
print('arms this pass:', [(variant, seed) for variant, seed, _, _ in JOBS])


arms this pass: [('base', 0), ('warm', 0), ('base', 1), ('warm', 1), ('base', 2), ('warm', 2)]


## 8. Preflight and bind the source artifacts

Every arm is checked against the corresponding 013 environment. Warm starts
are bound to that seed's SFT adapter and the merged base already used by 013 and
verified again during 022.


In [9]:
import glob, hashlib, json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from checkpoint import _config_hash
from config import load_config
from obs import _git_info, _versions
from train.model_utils import resolve_dtype
from train.sft import resolve_sft_start

D = '/content/drive/MyDrive/inm363'

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# reuse the existing 014 raw-base probe instead of touching the test set again
base_source = os.path.join(D, 'results', 'base_evals', '014_p8_COLAB_FULL_B7')
probe_files = ('base_logprob.json', 'base_logprob_preds.jsonl')
source_hashes = {}
for fname in probe_files:
    src = os.path.join(base_source, fname)
    assert os.path.exists(src), f'014 base probe file missing: {src}'
    source_hashes[fname] = sha(src)
source_probe = json.load(open(os.path.join(base_source, 'base_logprob.json')))
assert source_probe['n_items'] == 1884
with open(os.path.join(base_source, 'base_logprob_preds.jsonl')) as f:
    assert sum(1 for line in f if line.strip()) == 1884
base_source_manifest = json.load(open(os.path.join(
    D, 'results', '014_p8_COLAB_FULL_B7_sft_only_seed0', 'manifest.json')))
assert base_source_manifest['model_id'] == 'Qwen/Qwen2.5-7B'
assert base_source_manifest['resolved_dtype'] == 'bfloat16'
for dataset in ('cais/mmlu@', 'openai/gsm8k@'):
    assert base_source_manifest['versions'].get(dataset) == _versions().get(dataset)

reuse_record = {'source': 'results/base_evals/014_p8_COLAB_FULL_B7',
                'files': source_hashes, 'n_items': 1884,
                'source_manifest_run': '014_p8_COLAB_FULL_B7_sft_only_seed0',
                'source_git_commit': base_source_manifest['git']['commit'],
                'scope': 'raw 7b base probe, generative base eval not reused'}
for experiment in ('024_p16', '024_wsft_p16'):
    dest = os.path.join(D, 'results', 'base_evals', f'{experiment}_COLAB_FULL_B7')
    os.makedirs(dest, exist_ok=True)
    for fname in probe_files:
        src, dst = os.path.join(base_source, fname), os.path.join(dest, fname)
        if os.path.exists(dst):
            assert sha(dst) == source_hashes[fname], f'{experiment}: existing base probe differs from 014'
        else:
            tmp = dst + '.tmp'
            shutil.copy2(src, tmp)
            assert sha(tmp) == source_hashes[fname]
            os.replace(tmp, dst)
    reuse_path = os.path.join(dest, 'base_probe_reuse.json')
    if os.path.exists(reuse_path):
        assert json.load(open(reuse_path)) == reuse_record, f'{experiment}: reuse record changed'
    else:
        tmp = reuse_path + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(reuse_record, f, indent=2)
        os.replace(tmp, reuse_path)
    print(f'{experiment}: 014 base probe reused and hash-verified')

# bind both staged passes to the same notebook and scientific code
with open('notebooks/024_colab_b7_mc_sft_p16.ipynb') as f:
    notebook = json.load(f)
protocol_source = ''.join(
    line for cell in notebook['cells'] for line in cell.get('source', [])
    if 'SEEDS_TO_RUN =' not in line)
scientific_paths = ['config.py', 'checkpoint.py', 'enums.py', 'obs.py',
                    'scripts/build_data.py', 'scripts/env_switch.py',
                    'scripts/run.py', 'scripts/eval_adapter.py']
for folder in ('analysis', 'data', 'eval', 'train'):
    scientific_paths.extend(
        os.path.relpath(path) for path in glob.glob(f'{folder}/**/*.py', recursive=True))
scientific_paths = sorted(set(scientific_paths))
all_024_configs = sorted(glob.glob('configs/024*_colab_b7_seed*.yaml'))
execution_protocol = {
    'notebook_protocol_sha256': hashlib.sha256(protocol_source.encode()).hexdigest(),
    'scientific_file_hashes': {path: sha(path) for path in scientific_paths},
    'config_hashes': {path: _config_hash(load_config(path)) for path in all_024_configs}}
execution_sha = hashlib.sha256(
    json.dumps(execution_protocol, sort_keys=True).encode()).hexdigest()
execution_path = os.path.join(D, 'results', '024_execution_binding.json')
os.makedirs(os.path.dirname(execution_path), exist_ok=True)
if os.path.exists(execution_path):
    saved_execution = json.load(open(execution_path))
    assert saved_execution['protocol'] == execution_protocol, '024 scientific code changed between staged passes'
else:
    saved_execution = {'created_git': _git_info(), 'protocol_sha256': execution_sha,
                       'protocol': execution_protocol}
    tmp = execution_path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(saved_execution, f, indent=2)
    os.replace(tmp, execution_path)
assert saved_execution['protocol_sha256'] == execution_sha
print('024 staged execution binding verified')

for variant, seed, cfg_path, tag in JOBS:
    cfg = load_config(cfg_path)
    assert cfg.train.seed == seed and cfg.train.max_steps == 245
    assert cfg.train.sft_batch_size * cfg.train.sft_grad_accum == 16
    assert cfg.exec.allow_oom_fallback is False
    assert cfg.log.wandb_project == 'inm363-grpo'

    parent_tag = f'013_COLAB_FULL_B7_sft_grpo_seed{seed}'
    parent_result = os.path.join(D, 'results', parent_tag)
    parent_ckpt = os.path.join(D, 'checkpoints', parent_tag)
    parent = json.load(open(os.path.join(parent_result, 'manifest.json')))
    current_versions = _versions()
    for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate', 'cais/mmlu@', 'openai/gsm8k@'):
        assert current_versions.get(lib) == parent['versions'].get(lib), f'{tag}: {lib} differs from 013'
    assert resolve_dtype(cfg) == parent['resolved_dtype'], f'{tag}: dtype differs from 013'

    source = {'variant': variant, 'seed': seed, 'config': cfg_path,
              'config_hash': _config_hash(cfg), 'parent_run': parent_tag,
              'execution_protocol_sha256': execution_sha,
              'versions': {k: current_versions.get(k) for k in
                           ('torch', 'trl', 'transformers', 'peft', 'accelerate',
                            'cais/mmlu@', 'openai/gsm8k@')},
              'resolved_dtype': resolve_dtype(cfg)}
    if variant == 'warm':
        merged_config = os.path.join(parent_ckpt, 'sft_merged', 'config.json')
        assert os.path.exists(merged_config), f'{tag}: parent merged base is missing'
    model_src, base_kind, base_value = resolve_sft_start(cfg)
    if variant == 'base':
        assert not cfg.train.sft_from_adapter and base_kind == 'hf_id'
        source.update({'start_kind': base_kind, 'start_value': base_value})
    else:
        expected_adapter = f'checkpoints/{parent_tag}/sft'
        expected_merged = f'checkpoints/{parent_tag}/sft_merged'
        assert cfg.train.sft_from_adapter == expected_adapter
        assert base_kind == 'local' and base_value == expected_merged
        grpo_prov = json.load(open(os.path.join(parent_ckpt, 'grpo', 'adapter_base.json')))
        assert grpo_prov == {'kind': 'local', 'value': expected_merged}

        recorded = json.load(open(os.path.join(
            D, 'results', f'022_smooth_B7_seed{seed}_step1000', 'arm_manifest.json')))['hashes']
        merged_hashes = {}
        for rel, expected_hash in sorted(recorded.items()):
            src = os.path.join(parent_ckpt, rel)
            actual_hash = sha(src)
            assert actual_hash == expected_hash, f'{tag}: source base file drifted: {rel}'
            merged_hashes[rel] = actual_hash
        adapter_hashes = {}
        for fname in ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json'):
            adapter_hashes[fname] = sha(os.path.join(parent_ckpt, 'sft', fname))
        source.update({'start_kind': base_kind, 'start_value': base_value,
                       'adapter_hashes': adapter_hashes, 'merged_hashes': merged_hashes})

    out = os.path.join(D, 'results', tag)
    os.makedirs(out, exist_ok=True)
    source_path = os.path.join(out, 'source_binding.json')
    if os.path.exists(source_path):
        assert json.load(open(source_path)) == source, f'{tag}: source binding changed'
    else:
        with open(source_path, 'w') as f:
            json.dump(source, f, indent=2)
    print(f'{tag}: source and environment verified')


024_p16: 014 base probe reused and hash-verified
024_wsft_p16: 014 base probe reused and hash-verified
024 staged execution binding verified
024_p16_COLAB_FULL_B7_sft_only_seed0: source and environment verified
024_wsft_p16_COLAB_FULL_B7_sft_only_seed0: source and environment verified
024_p16_COLAB_FULL_B7_sft_only_seed1: source and environment verified
024_wsft_p16_COLAB_FULL_B7_sft_only_seed1: source and environment verified
024_p16_COLAB_FULL_B7_sft_only_seed2: source and environment verified
024_wsft_p16_COLAB_FULL_B7_sft_only_seed2: source and environment verified


## 9. Wiring check


In [10]:
import subprocess, sys
for _, _, cfg_path, tag in JOBS:
    print('wiring:', tag)
    subprocess.run([sys.executable, 'scripts/run.py', '--config', cfg_path], check=True)


wiring: 024_p16_COLAB_FULL_B7_sft_only_seed0
wiring: 024_wsft_p16_COLAB_FULL_B7_sft_only_seed0
wiring: 024_p16_COLAB_FULL_B7_sft_only_seed1
wiring: 024_wsft_p16_COLAB_FULL_B7_sft_only_seed1
wiring: 024_p16_COLAB_FULL_B7_sft_only_seed2
wiring: 024_wsft_p16_COLAB_FULL_B7_sft_only_seed2


## 10. Train the selected arms and run the probes

The normal runner is resume-safe. Generative evaluation stays off here and is
run only after the training gates pass.


In [11]:
import os
from checkpoint import _config_hash
from config import load_config

def training_probe_complete(cfg_path, tag):
    cfg = load_config(cfg_path)
    out = os.path.join(D, 'results', tag)
    adapter = os.path.join(D, 'checkpoints', tag, 'sft')
    needed = [os.path.join(adapter, name) for name in
              ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json',
               'config_hash.txt')]
    needed.extend(os.path.join(out, name) for name in
                  ('sft_trainlog.json', 'sft_logprob.json', 'sft_logprob_preds.jsonl',
                   'manifest.json', 'summary.json'))
    if not all(os.path.exists(path) for path in needed):
        return False
    return open(os.path.join(adapter, 'config_hash.txt')).read().strip() == _config_hash(cfg)

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in section 0'
for variant, seed, cfg_path, tag in JOBS:
    if training_probe_complete(cfg_path, tag):
        print(f'{tag}: training and probe complete, skipped')
        continue
    print(f'===== train {tag} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config {cfg_path} --stage all
    if _exit_code != 0:
        raise SystemExit(f'{tag} failed with exit code {_exit_code}; keep the declared settings unchanged')


024_p16_COLAB_FULL_B7_sft_only_seed0: training and probe complete, skipped
024_wsft_p16_COLAB_FULL_B7_sft_only_seed0: training and probe complete, skipped
===== train 024_p16_COLAB_FULL_B7_sft_only_seed1 =====
2026-08-01 17:04:50,602 INFO using wandb key from env
2026-08-01 17:04:50,603 INFO run 024_p16_COLAB_FULL_B7_sft_only_seed1 device=cuda stages=['base_eval', 'sft', 'sft_eval'] trackers=['tensorboard', 'wandb'] out=/content/drive/MyDrive/inm363/results/024_p16_COLAB_FULL_B7_sft_only_seed1
2026-08-01 17:04:50,603 INFO base_eval skipped, probe-only run
2026-08-01 17:04:50,604 INFO base_logprob_eval already done, skipping
config.json: 100% 686/686 [00:00<00:00, 3.66MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 21.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 93.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 110MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 143MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 73.6MB/s]
Reconstructing (incomp

## 11. Training and provenance gates


In [12]:
import glob, json, os
from collections import Counter
from checkpoint import _config_hash
from config import load_config

D = '/content/drive/MyDrive/inm363'
for variant, seed, cfg_path, tag in JOBS:
    cfg = load_config(cfg_path)
    result_dir = os.path.join(D, 'results', tag)
    adapter_dir = os.path.join(D, 'checkpoints', tag, 'sft')
    for fname in ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json'):
        assert os.path.exists(os.path.join(adapter_dir, fname)), f'{tag}: missing {fname}'
    hash_path = os.path.join(adapter_dir, 'config_hash.txt')
    assert os.path.exists(hash_path) and open(hash_path).read().strip() == _config_hash(cfg)

    history = json.load(open(os.path.join(result_dir, 'sft_trainlog.json')))
    steps = [(int(row['step']), row) for row in history
             if 'step' in row and 'learning_rate' in row and 'loss' in row]
    counts = Counter(step for step, _ in steps)
    assert len(steps) == 245 and set(counts) == set(range(1, 246)), f'{tag}: step coverage is not 1..245'
    assert all(n == 1 for n in counts.values()), f'{tag}: duplicate training steps found'
    lrs = [row['learning_rate'] for _, row in sorted(steps)]
    assert 9.9e-5 <= lrs[0] <= 1.0e-4 and lrs[-1] <= 5e-7
    assert all(b <= a + 1e-12 for a, b in zip(lrs, lrs[1:])), f'{tag}: learning rate increased'

    prov = json.load(open(os.path.join(adapter_dir, 'adapter_base.json')))
    if variant == 'base':
        assert prov == {'kind': 'hf_id', 'value': 'Qwen/Qwen2.5-7B'}
    else:
        expected = f'checkpoints/013_COLAB_FULL_B7_sft_grpo_seed{seed}/sft_merged'
        assert prov == {'kind': 'local', 'value': expected}, f'{tag}: wrong warm-start provenance'

    probe = json.load(open(os.path.join(result_dir, 'sft_logprob.json')))
    assert probe['n_items'] == 1884
    with open(os.path.join(result_dir, 'sft_logprob_preds.jsonl')) as f:
        assert sum(1 for line in f if line.strip()) == 1884
    manifest = json.load(open(os.path.join(result_dir, 'manifest.json')))
    assert manifest['seed'] == seed and manifest['config']['train']['max_steps'] == 245
    print(f'{tag}: 245 steps, schedule, probe and provenance passed')


024_p16_COLAB_FULL_B7_sft_only_seed0: 245 steps, schedule, probe and provenance passed
024_wsft_p16_COLAB_FULL_B7_sft_only_seed0: 245 steps, schedule, probe and provenance passed
024_p16_COLAB_FULL_B7_sft_only_seed1: 245 steps, schedule, probe and provenance passed
024_wsft_p16_COLAB_FULL_B7_sft_only_seed1: 245 steps, schedule, probe and provenance passed
024_p16_COLAB_FULL_B7_sft_only_seed2: 245 steps, schedule, probe and provenance passed
024_wsft_p16_COLAB_FULL_B7_sft_only_seed2: 245 steps, schedule, probe and provenance passed


## 12. Generative evaluations

Each evaluation is bound to the final adapter before any output is accepted.
Both prediction files must contain all 1,884 frozen test items.


In [13]:
import hashlib, json, os

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
for variant, seed, cfg_path, tag in JOBS:
    out = os.path.join(D, 'results', tag)
    adapter = os.path.join(D, 'checkpoints', tag, 'sft')
    binding = {name: sha(os.path.join(adapter, name)) for name in
               ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json')}
    bind_path = os.path.join(out, 'sft_eval_binding.json')
    generative_exists = any(os.path.exists(os.path.join(out, name)) for name in
                            ('sft_eval.json', 'sft_preds.jsonl'))
    if os.path.exists(bind_path):
        assert json.load(open(bind_path)) == binding, f'{tag}: evaluation binding changed'
    else:
        assert not generative_exists, f'{tag}: generative outputs exist without a binding'
        with open(bind_path, 'w') as f:
            json.dump(binding, f, indent=2)

    print(f'===== evaluate {tag} =====')
    !python scripts/eval_adapter.py --config {cfg_path} --adapter {adapter} --condition sft --out {out} --seed {seed}
    if _exit_code != 0:
        raise SystemExit(f'{tag} evaluation failed with exit code {_exit_code}')

    ev = json.load(open(os.path.join(out, 'sft_eval.json')))
    lp = json.load(open(os.path.join(out, 'sft_logprob.json')))
    assert ev['n_items'] == 1884 and lp['n_items'] == 1884
    for pred_file in ('sft_preds.jsonl', 'sft_logprob_preds.jsonl'):
        with open(os.path.join(out, pred_file)) as f:
            rows = sum(1 for line in f if line.strip())
        assert rows == 1884, f'{tag}: {pred_file} has {rows} rows'
    eval_manifest = json.load(open(os.path.join(out, 'sft_eval_manifest.json')))
    assert eval_manifest.get('completed') is True
    print(f'{tag}: evaluation complete and bound')


===== evaluate 024_p16_COLAB_FULL_B7_sft_only_seed0 =====
sft gen eval already done, skipping
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.8873720136518771, 'medium': 0.7783251231527094, 'hard': 0.5580808080808081}
probe per-tier: {'easy': 0.89419795221843, 'medium': 0.80623973727422, 'hard': 0.5757575757575758}
024_p16_COLAB_FULL_B7_sft_only_seed0: evaluation complete and bound
===== evaluate 024_wsft_p16_COLAB_FULL_B7_sft_only_seed0 =====
sft gen eval already done, skipping
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.875995449374289, 'medium': 0.7963875205254516, 'hard': 0.5328282828282829}
probe per-tier: {'easy': 0.9010238907849829, 'medium': 0.812807881773399, 'hard': 0.5555555555555556}
024_wsft_p16_COLAB_FULL_B7_sft_only_seed0: evaluation complete and bound
===== evaluate 024_p16_COLAB_FULL_B7_sft_only_seed1 =====
Loading weights: 100% 339/339 [00:00<00:00, 5846.65it/s]
sft eval 

## 13. Registered comparison

The binding result appears only when all six new arms are complete. The two
three-seed comparison families use Holm-adjusted exact McNemar tests. Material
change requires at least 2 points and adjusted p below .05; equivalence requires
the paired 95% interval to stay inside plus or minus 2 points. Otherwise the arm
is inconclusive.

The p8 warm-start comparison is exact only for seed 0 because experiment 015
used the seed-0 GSM8K adapter for all three seeds. Seeds 1 and 2 are kept
descriptive for that comparison.


In [14]:
import json, os, sys
sys.path.insert(0, os.getcwd())
from analysis.stats import _read_preds, align_pred_rows, holm, mcnemar_exact, paired_delta_ci
from data.subjects import TIERS, tier_of

D = '/content/drive/MyDrive/inm363'

def preds(tag, condition):
    return _read_preds(os.path.join(D, 'results', tag, f'{condition}_preds.jsonl'))

def metric(new_rows, ref_rows):
    new_aligned, ref_aligned = align_pred_rows(new_rows, ref_rows)
    new_correct = [bool(row['correct']) for row in new_aligned]
    ref_correct = [bool(row['correct']) for row in ref_aligned]
    delta, lo, hi = paired_delta_ci(ref_correct, new_correct)
    n01 = sum(not a and b for a, b in zip(ref_correct, new_correct))
    n10 = sum(a and not b for a, b in zip(ref_correct, new_correct))
    tiers = {}
    for tier in TIERS:
        pairs = [(a, b) for a, b, row in zip(ref_correct, new_correct, new_aligned)
                 if tier_of(row['subject']) == tier]
        tiers[tier] = {'reference': sum(a for a, _ in pairs) / len(pairs),
                       'new': sum(b for _, b in pairs) / len(pairs)}
    return {'reference_acc': sum(ref_correct) / len(ref_correct),
            'new_acc': sum(new_correct) / len(new_correct),
            'delta': delta, 'ci': [lo, hi], 'p_raw': mcnemar_exact(n01, n10),
            'tiers': tiers}

def classify(row, adjusted_p):
    if abs(row['delta']) >= 0.02 and adjusted_p < 0.05:
        return 'material improvement' if row['delta'] > 0 else 'material decline'
    if row['ci'][0] > -0.02 and row['ci'][1] < 0.02:
        return 'equivalent'
    return 'inconclusive'

def complete(tag):
    root = os.path.join(D, 'results', tag)
    return all(os.path.exists(os.path.join(root, name)) for name in
               ('sft_eval.json', 'sft_preds.jsonl', 'sft_logprob.json', 'sft_logprob_preds.jsonl'))

new_tags = [(f'024_p16_COLAB_FULL_B7_sft_only_seed{s}',
             f'024_wsft_p16_COLAB_FULL_B7_sft_only_seed{s}') for s in (0, 1, 2)]
binding = all(complete(tag) for pair in new_tags for tag in pair)
done = [s for s, pair in enumerate(new_tags) if all(complete(tag) for tag in pair)]
print('complete seed pairs:', done)
if not binding:
    print('PROVISIONAL: registered categories wait for all six arms')

rows = []
for seed in done:
    base_new = preds(f'024_p16_COLAB_FULL_B7_sft_only_seed{seed}', 'sft')
    warm_new = preds(f'024_wsft_p16_COLAB_FULL_B7_sft_only_seed{seed}', 'sft')
    base_p8 = preds(f'014_p8_COLAB_FULL_B7_sft_only_seed{seed}', 'sft')
    warm_p8 = preds(f'015_wsft_COLAB_FULL_B7_sft_only_seed{seed}', 'sft')
    grpo = preds(f'022_smooth_B7_seed{seed}_step1000', 'grpo')

    base_budget = metric(base_new, base_p8)
    warm_vs_grpo = metric(warm_new, grpo)
    warm_budget = metric(warm_new, warm_p8)
    warm_budget['exact_preparation_match'] = seed == 0

    base_probe = preds(f'024_p16_COLAB_FULL_B7_sft_only_seed{seed}', 'sft_logprob')
    warm_probe = preds(f'024_wsft_p16_COLAB_FULL_B7_sft_only_seed{seed}', 'sft_logprob')
    base_start_probe = _read_preds(os.path.join(
        D, 'results', 'base_evals', '024_p16_COLAB_FULL_B7', 'base_logprob_preds.jsonl'))
    warm_start_probe = preds(f'013_COLAB_FULL_B7_sft_grpo_seed{seed}', 'sft_logprob')
    base_probe_change = metric(base_probe, base_start_probe)
    warm_probe_change = metric(warm_probe, warm_start_probe)

    base_eval = json.load(open(os.path.join(
        D, 'results', f'024_p16_COLAB_FULL_B7_sft_only_seed{seed}', 'sft_eval.json')))
    warm_eval = json.load(open(os.path.join(
        D, 'results', f'024_wsft_p16_COLAB_FULL_B7_sft_only_seed{seed}', 'sft_eval.json')))
    base_hard_margin = (base_budget['tiers']['hard']['new']
                        - base_probe_change['tiers']['hard']['new'])
    warm_hard_margin = (warm_vs_grpo['tiers']['hard']['new']
                        - warm_probe_change['tiers']['hard']['new'])
    rows.append({'seed': seed, 'base_budget': base_budget,
                 'warm_vs_smooth_grpo': warm_vs_grpo, 'warm_budget_descriptive': warm_budget,
                 'base_probe_from_start': base_probe_change,
                 'warm_probe_from_start': warm_probe_change,
                 'endpoints': {'base_p16': {'overall_acc': base_budget['new_acc'],
                                            'hard_acc': base_budget['tiers']['hard']['new'],
                                            'probe_acc': base_probe_change['new_acc'],
                                            'hard_probe': base_probe_change['tiers']['hard']['new'],
                                            'hard_margin': base_hard_margin,
                                            'format_compliance': base_eval['format_compliance'],
                                            'extraction_failure_rate': base_eval['extraction_failure_rate'],
                                            'mean_len': base_eval['mean_len']},
                               'warm_p16': {'overall_acc': warm_vs_grpo['new_acc'],
                                            'hard_acc': warm_vs_grpo['tiers']['hard']['new'],
                                            'probe_acc': warm_probe_change['new_acc'],
                                            'hard_probe': warm_probe_change['tiers']['hard']['new'],
                                            'hard_margin': warm_hard_margin,
                                            'format_compliance': warm_eval['format_compliance'],
                                            'extraction_failure_rate': warm_eval['extraction_failure_rate'],
                                            'mean_len': warm_eval['mean_len']}}})

families = ('base_budget', 'warm_vs_smooth_grpo')
common = {}
if binding:
    for family in families:
        adjusted = holm([row[family]['p_raw'] for row in rows])
        categories = []
        for row, p_adj in zip(rows, adjusted):
            row[family]['holm_p'] = p_adj
            row[family]['category'] = classify(row[family], p_adj)
            categories.append(row[family]['category'])
        if (len(set(categories)) == 1
                and categories[0] in ('equivalent', 'material improvement', 'material decline')):
            common[family] = categories[0]
        else:
            common[family] = 'mixed or inconclusive, report arm by arm'
else:
    for row in rows:
        for family in families:
            row[family]['holm_p'] = None
            row[family]['category'] = 'PROVISIONAL'

for row in rows:
    seed = row['seed']
    for family in families:
        result = row[family]
        print(f"seed {seed} {family}: {result['reference_acc']:.3f} -> {result['new_acc']:.3f} "
              f"({result['delta']*100:+.1f}pt, ci [{result['ci'][0]*100:+.1f}, "
              f"{result['ci'][1]*100:+.1f}], category {result['category']})")
        for tier, values in result['tiers'].items():
            print(f"   {tier}: {values['reference']:.3f} -> {values['new']:.3f}")
    wb = row['warm_budget_descriptive']
    label = 'exact preparation match' if wb['exact_preparation_match'] else 'different warm-start preparation'
    print(f"seed {seed} warm p8 -> p16: {wb['delta']*100:+.1f}pt ({label}, descriptive)")
    for probe_name in ('base_probe_from_start', 'warm_probe_from_start'):
        probe = row[probe_name]
        probe_class = ('unchanged' if probe['ci'][0] > -0.02 and probe['ci'][1] < 0.02
                       else 'equivalence not established')
        probe['classification'] = probe_class
        print(f"   {probe_name}: {probe['delta']*100:+.1f}pt "
              f"ci [{probe['ci'][0]*100:+.1f}, {probe['ci'][1]*100:+.1f}] -> {probe_class}")

summary = {'binding': binding, 'complete_seeds': done, 'common_readings': common, 'arms': rows,
           'scope': {'base_budget': '024_p16 against 014_p8, same seed',
                     'warm_vs_smooth_grpo': '024 warm p16 against 022 smooth-1000, same 013 start and seed',
                     'warm_budget': 'exact preparation match only at seed 0; seeds 1 and 2 descriptive',
                     'schedule': 'each budget uses its own linear decay, method as practised'}}
with open(os.path.join(D, 'results', '024_comparison_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('comparison saved, binding =', binding)


complete seed pairs: [0, 1, 2]
seed 0 base_budget: 0.786 -> 0.783 (-0.3pt, ci [-1.2, +0.5], category equivalent)
   easy: 0.895 -> 0.887
   medium: 0.780 -> 0.778
   hard: 0.553 -> 0.558
seed 0 warm_vs_smooth_grpo: 0.808 -> 0.778 (-3.0pt, ci [-4.5, -1.4], category material decline)
   easy: 0.892 -> 0.876
   medium: 0.795 -> 0.796
   hard: 0.641 -> 0.533
seed 0 warm p8 -> p16: +0.4pt (exact preparation match, descriptive)
   base_probe_from_start: -0.2pt ci [-1.0, +0.6] -> unchanged
   warm_probe_from_start: -0.1pt ci [-1.0, +0.8] -> unchanged
seed 1 base_budget: 0.780 -> 0.784 (+0.4pt, ci [-0.5, +1.2], category equivalent)
   easy: 0.889 -> 0.890
   medium: 0.778 -> 0.791
   hard: 0.543 -> 0.538
seed 1 warm_vs_smooth_grpo: 0.801 -> 0.782 (-2.0pt, ci [-3.6, -0.4], category inconclusive)
   easy: 0.889 -> 0.884
   medium: 0.782 -> 0.788
   hard: 0.639 -> 0.545
seed 1 warm p8 -> p16: +0.5pt (different warm-start preparation, descriptive)
   base_probe_from_start: -0.3pt ci [-1.1, +0.6] -

## 14. Close the units record


In [ ]:
# close the compute-unit record for this session
import json, os, time
cu_after = 63.6   # fill from the resources panel
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run section 0 first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)


reminder: fill cu_after for the record
